# Bank Marketing: Understanding and Predicting Term Deposits

This project uses a bank marketing dataset to explore which customer and campaign details are linked with term-deposit subscriptions. The notebook keeps the workflow simple: inspect the data, make a few charts, train two basic models, and compare their results.

**Before running:** put `bank.csv` in the same folder as this notebook. The target column should be named `deposit` or `y` and contain `yes`/`no` values. This project is a learning exercise, not a real banking decision system.


## 1. Import libraries

These libraries are used for data handling, charts, and basic machine learning.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay


## 2. Load the data

The reader below tries to detect whether the CSV uses commas or semicolons. If your file is stored somewhere else, update `DATA_PATH`.

In [ ]:
DATA_PATH = "bank.csv"
df = pd.read_csv(DATA_PATH, sep=None, engine="python")
df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumns:", df.columns.tolist())


## 3. Check the data

Look at column types, missing values, and the target balance before modelling.

In [ ]:
df.info()
print("\nMissing values per column:")
display(df.isna().sum().sort_values(ascending=False).head(15))

TARGET = "deposit" if "deposit" in df.columns else "y" if "y" in df.columns else None
if TARGET is None:
    raise ValueError("Could not find target column. Expected a column named 'deposit' or 'y'.")

print("\nTarget column:", TARGET)
display(df[TARGET].value_counts(dropna=False))


## 4. Explore subscription patterns

First, compare the number of customers who subscribed with the number who did not.

In [ ]:
df[TARGET].astype(str).str.lower().value_counts().plot(kind="bar", rot=0)
plt.title("Term-deposit subscriptions")
plt.xlabel("Subscribed?")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()


Now look at subscription rates by month and job, when those columns are available. These charts show patterns in this dataset; they do not prove that a particular group will subscribe.

In [ ]:
df_eda = df.copy()
df_eda["subscribed"] = df_eda[TARGET].astype(str).str.lower().map({"yes": 1, "no": 0})

for column in ["month", "job"]:
    if column in df_eda.columns:
        rates = df_eda.groupby(column, dropna=False)["subscribed"].mean().sort_values()
        rates.plot(kind="barh", figsize=(8, max(3, len(rates) * 0.28)))
        plt.title(f"Subscription rate by {column}")
        plt.xlabel("Share who subscribed")
        plt.ylabel(column.capitalize())
        plt.tight_layout()
        plt.show()


## 5. Prepare data for machine learning

The target is converted to 1 (`yes`) and 0 (`no`). We remove `duration` because call length is only known after the call; including it would make this less useful for deciding whom to contact before a call. Missing values and categorical columns are handled inside a preprocessing pipeline.

In [ ]:
y = df[TARGET].astype(str).str.lower().map({"yes": 1, "no": 0})
if y.isna().any():
    raise ValueError("The target column must contain only 'yes' and 'no' values.")

drop_columns = [TARGET]
if "duration" in df.columns:
    drop_columns.append("duration")
X = df.drop(columns=drop_columns).copy()

categorical_columns = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_columns = X.select_dtypes(include=["number"]).columns.tolist()

print("Categorical features:", len(categorical_columns))
print("Numeric features:", len(numeric_columns))
print("Duration excluded from prediction:", "duration" in drop_columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_columns),
    ("cat", categorical_pipeline, categorical_columns)
])


## 6. Train two models

Logistic Regression is a straightforward baseline. Random Forest can capture more complex patterns. Both use the same preprocessing steps so the comparison is fairer and the code stays manageable.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=150, random_state=42, class_weight="balanced", n_jobs=-1
    )
}

fitted_models = {}
results = []

for name, estimator in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    fitted_models[name] = pipeline

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1-score", ascending=False)
display(results_df.round(3))


## 7. View the confusion matrix

Choose a model by considering precision and recall as well as accuracy. The cell below shows the confusion matrix for the model with the higher F1-score in this test comparison.

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_name]
best_predictions = best_model.predict(X_test)

print("Model shown:", best_name)
ConfusionMatrixDisplay.from_predictions(
    y_test, best_predictions,
    display_labels=["Did not subscribe", "Subscribed"],
    values_format="d"
)
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()


## 8. What features mattered to the Random Forest?

This chart gives a simple view of which encoded features the Random Forest relied on most. These are model associations, not proof that a feature causes a subscription.

In [ ]:
rf_pipeline = fitted_models["Random Forest"]
rf_model = rf_pipeline.named_steps["model"]
feature_names = rf_pipeline.named_steps["preprocessor"].get_feature_names_out()

importance = pd.Series(rf_model.feature_importances_, index=feature_names)
top_features = importance.nlargest(10).sort_values()
top_features.plot(kind="barh", figsize=(8, 5))
plt.title("Top 10 Random Forest feature importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 9. Wrap-up

Use the actual charts and metric table above to write a short conclusion. Mention which model performed better on this test split, what the precision/recall trade-off looked like, and one or two patterns from the EDA. Avoid claiming the results will generalize to every bank or customer group.

**Limitations:** this is a public practice dataset, the test score can change with the data split, and feature importance shows association rather than causation. A real campaign would need additional validation before the model could be used.